# Gemma 4 E4B-it — Interactive inference sandbox

Loads `google/gemma-4-E4B-it` via the canonical HuggingFace pattern
(`AutoModelForCausalLM` + `AutoProcessor`, `dtype='auto'`) and exposes a single
editable cell with `IMAGE_PATH` + `PROMPT_ID` so you can iterate prompt by
prompt without reloading the model.

Mirror of `paligemma2_3b_mix_448_inference_test.ipynb` so you can compare the
two families on the same image with the same four prompts (one per scenario
the production app recognises).

## How to use

1. **Runtime → A100** (E4B in bfloat16 needs ~16 GB VRAM; T4 free won't fit).
2. Run **§1** (install) and **§2** (load model). Model load takes ~50 s the
   first time and stays in VRAM until you restart the runtime.
3. Upload your test image to `/content/` (drag-and-drop in the file panel).
4. Edit `IMAGE_PATH` and `PROMPT_ID` in **§3** and run that cell. Re-run §3
   as many times as you like — §1 and §2 don't need to be re-executed.

## What this notebook does NOT do

It does not score the answer, log to CSV, or run multiple prompts at once.
It's a manual exploration tool. For systematic evaluation use
`experiment1_gemma.ipynb` / `experiment2_gemma.ipynb`.


## 1. Install dependencies

If this is a fresh runtime, run this cell once and then `Runtime → Restart
runtime` (Pillow gets force-reinstalled, the kernel needs to reload it). On
restart you can skip §1 and start at §2.


In [ ]:
!pip install -qU --force-reinstall Pillow
!pip install -qU transformers accelerate
print()
print('=' * 60)
print('FIRST RUN ON FRESH RUNTIME: Runtime -> Restart runtime now,')
print('then run cells from sec 2 onwards (do NOT re-run sec 1).')
print('=' * 60)


## 2. Load model and processor

Loads Gemma 4 E4B-it once. Uses `dtype='auto'` (the canonical pattern from
the Gemma 4 model card) which resolves to bfloat16 on A100 and float32 on
CPU. After this cell completes, `model` and `processor` stay in memory until
the runtime restarts — so §3 can be edited and re-run cheaply.

Gated model: set `HF_TOKEN` below with a token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) that
has accepted the [model terms](https://huggingface.co/google/gemma-4-E4B-it).


In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

HF_TOKEN = ''  # paste your HuggingFace read token here
MODEL_ID = 'google/gemma-4-E4B-it'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {DEVICE} (dtype=auto)...')
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype='auto' if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded -- {n_params / 1e9:.2f}B params')
if DEVICE == 'cuda':
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')


## 3. Inference (edit IMAGE_PATH and PROMPT_ID, re-run this cell)

The four prompts mirror exactly what the on-device VLM (`vlm_smoke_test`) and
the PaliGemma sandbox use, so the output here is directly comparable across
families on the same image:

| `PROMPT_ID` | Use when image shows... | Expected output format |
|---|---|---|
| `door_pddl` | A classroom door plate (e.g. `1.7`, `IA`) | `(at user1 classroom_1_7)` |
| `sign_pddl` | A directory sign in building G | `(at user1 directory_sign_g_0)` |
| `stairs_text` | A staircase | `stairs up` or `stairs down` |
| `elevator_json_cabin` | An elevator (doors visible) | `{"type": "elevator", "detail": "open"}` |

Unlike PaliGemma (which takes a raw `<image>{prompt}`), Gemma 4 expects a
chat-template input with `system` + `user` messages, where the user content
is a list of `{image, text}` blocks (image first per Gemma 4's modality-order
best practice). A short system prompt nudges the model towards format-faithful
responses, matching the intent of PaliGemma's terse VQA-style outputs.

Decoding stays greedy (`do_sample=False, num_beams=1`, `enable_thinking=False`)
so the response is deterministic for the chosen (image, prompt) pair.
Re-run §3 as many times as you want — `model` and `processor` stay in VRAM.


In [ ]:
# === EDIT THESE TWO ===
IMAGE_PATH = '/content/your_image.jpg'  # drag-and-drop your image into /content/

# Pick one of: 'door_pddl', 'sign_pddl', 'stairs_text', 'elevator_json_cabin'
PROMPT_ID = 'door_pddl'

MAX_NEW_TOKENS = 60
# ======================

SYSTEM_PROMPT = (
    'You are a visual assistant for indoor navigation. '
    'Answer concisely and exactly in the format requested by the question. '
    'Do not add explanations or extra words.'
)

# Prompts copied verbatim from vlm_smoke_test/lib/prompts.dart so this notebook
# evaluates exactly what the on-device VLM is asked in production.
PROMPTS = {
    'door_pddl': (
        'What room number is on the plate in this image?\n'
        '\n'
        'Read the number carefully. It contains a DOT in the middle.\n'
        'The DOT separates two digits. Keep both digits and put _ where the DOT is.\n'
        '\n'
        'Plate says 3.7 -> output: (at user1 classroom_3_7)\n'
        'Plate says 0.2 -> output: (at user1 classroom_0_2)\n'
        'Plate says IA  -> output: (at user1 lab_ia)\n'
        '\n'
        'Output ONLY the (at user1 ...) line.'
    ),
    'sign_pddl': (
        'Look at this image. It shows a large white sign on a wall inside building G.\n'
        '\n'
        'In the top-right area of the sign there are two lines of text, one above the other:\n'
        '- Top line: the building letter (G)\n'
        '- Bottom line: either a floor number (0, 1, or 2) OR the word "Agora"\n'
        '\n'
        'Read the bottom line and map it to the answer:\n'
        '- "0" -> (at user1 directory_sign_g_0)\n'
        '- "1" -> (at user1 directory_sign_g_1)\n'
        '- "2" -> (at user1 directory_sign_g_2)\n'
        '- "Agora" -> (at user1 directory_sign_g_agora)\n'
        '\n'
        'Output ONLY the (at user1 ...) line. Nothing else.'
    ),
    'stairs_text': (
        'Look at this image. It shows a staircase.\n'
        '\n'
        'Question: from the point of view of the camera, do the stairs go UP or DOWN?\n'
        '\n'
        'How to decide:\n'
        '- If the next step is higher than the camera and the stairs rise away from you -> up\n'
        '- If the next step is lower than the camera and the stairs drop away from you -> down\n'
        '\n'
        'Answer with exactly one of these two phrases:\n'
        'stairs up\n'
        'stairs down\n'
        '\n'
        'Output ONLY the phrase. No punctuation, no extra words.'
    ),
    'elevator_json_cabin': (
        'Look at this image. Focus on the elevator doors.\n'
        '\n'
        'Can you see the inside of the elevator cabin?\n'
        '- YES, I can see inside the cabin -> the doors are open\n'
        '- NO, I cannot see inside, the doors are a solid surface -> the doors are closed\n'
        '\n'
        'Valid answers:\n'
        '{"type": "elevator", "detail": "open"}\n'
        '{"type": "elevator", "detail": "closed"}\n'
        '\n'
        'Output ONLY the JSON. Nothing else.'
    ),
}

if PROMPT_ID not in PROMPTS:
    raise ValueError(f'Unknown PROMPT_ID {PROMPT_ID!r}. Pick one of: {list(PROMPTS)}')
PROMPT = PROMPTS[PROMPT_ID]

image = Image.open(IMAGE_PATH).convert('RGB')

# Display the image inline so you can confirm it's the right one.
try:
    from IPython.display import display
    display(image.resize((min(448, image.width), min(448, image.height))))
except Exception:
    pass

# Build chat-template inputs. Image precedes text in user content per Gemma 4's
# modality-order best practice. System and user content are both LISTS of
# dicts (Gemma 3n/4 chat templates iterate content blocks expecting dicts with
# a 'type' field; passing a raw string for system content raises
# TypeError: string indices must be integers).
messages = [
    {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
    {'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text',  'text': PROMPT},
    ]},
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    enable_thinking=False,
).to(model.device)
# BatchFeature.to(dtype=...) only casts float tensors, so input_ids stay int64
# while pixel_values get the right model dtype.
inputs = inputs.to(dtype=model.dtype)

gen_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    num_beams=1,
    pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
)

import time
if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    output = model.generate(**inputs, **gen_kwargs)
if torch.cuda.is_available():
    torch.cuda.synchronize()
latency_ms = (time.time() - t0) * 1000

input_len = inputs['input_ids'].shape[-1]
response = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

print('=' * 60)
print(f'MODEL:     {MODEL_ID}')
print(f'PROMPT_ID: {PROMPT_ID}')
print(f'IMAGE:     {IMAGE_PATH}')
print('-' * 60)
print('SYSTEM PROMPT:')
print(SYSTEM_PROMPT)
print('-' * 60)
print('PROMPT:')
print(PROMPT)
print('-' * 60)
print('RESPONSE:')
print(response)
print('-' * 60)
print(f'Latency: {latency_ms:.1f} ms  |  Output tokens: {output.shape[1] - input_len}')
print('=' * 60)
